# 00 · Checkpoint provenance

Documents exactly which weights every reported number comes from.

For each checkpoint the notebook records the file hash, the training configuration, the step counters and the training history stored inside the checkpoint, and it compares the weights of every pair of checkpoints to detect duplicated or mislabelled files.

The training notebooks in `notebooks/archive/` were executed across several Kaggle sessions with checkpoint resume, so their cell outputs were not preserved. This notebook reconstructs the training logs from the `history` saved in each checkpoint.

**Kaggle inputs to attach:** the three reported checkpoints (`secunda-slakh-60-epochs`, `secunda-finetuned-on-slakh`, `secunda-finetuned-on-ost-v2`), the two extra model inputs used by older scripts (`secunda-finetuned-on-ost`, `secundafinetuned`), and the dataset `slakh-and-custom-csvs`. Missing extra inputs are reported and skipped.

In [ ]:
# ── Repository bootstrap: make src/ importable on Kaggle and locally ──
# On Kaggle the repository is cloned into /kaggle/working (Internet must be
# enabled in the notebook settings). Locally, set SECUNDA_REPO_DIR to the
# repository root and nothing is cloned.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DanBracs7/Adaptive-Secunda-Synth.git"
REPO_BRANCH = "revision"  # switch to "main" after merging the revision
REPO_DIR = Path(os.environ.get("SECUNDA_REPO_DIR", "/kaggle/working/Adaptive-Secunda-Synth"))

if not (REPO_DIR / "src").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "src"))

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit or "unknown (not a git checkout)")

In [ ]:
import hashlib
import importlib.metadata
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from paths import get_path

print("Python:", platform.python_version())

for package_name in ["torch", "torchaudio", "encodec", "librosa", "numpy", "pandas", "scipy"]:
    try:
        version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        version = "not installed"
    print(f"{package_name}: {version}")

print("CUDA available:", torch.cuda.is_available())

In [ ]:
OUTPUT_DIR = get_path("output_dir") / "provenance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoints reported in the paper, in chronological order.
CHECKPOINTS = {
    "Baseline": get_path("ckpt_baseline"),
    "Slakh-FT": get_path("ckpt_slakh_ft"),
    "OST-FT": get_path("ckpt_ost_ft"),
}

# Files referenced by older code, checked only for provenance:
#   OST-FT-v1: loaded by generation_test_8cb.py (first submission);
#   OST-init:  weights the OST fine-tune notebook was initialised from.
EXTRA_CHECKPOINTS = {
    "OST-FT-v1": Path(os.environ.get(
        "SECUNDA_CKPT_OST_FT_V1",
        "/kaggle/input/models/danielebracoloni/secunda-finetuned-on-ost/"
        "pytorch/default/1/secunda_slakh8cb_finetune_best.pt",
    )),
    "OST-init": Path(os.environ.get(
        "SECUNDA_CKPT_OST_INIT",
        "/kaggle/input/models/danielebracoloni/secundafinetuned/"
        "pytorch/default/1/secunda_slakh8cb_finetune_best.pt",
    )),
}

# Manifest copy mounted on Kaggle during training and evaluation.
KAGGLE_MANIFEST = Path(os.environ.get(
    "SECUNDA_KAGGLE_MANIFEST",
    "/kaggle/input/datasets/danielebracoloni/slakh-and-custom-csvs/all_tracks.csv",
))

In [ ]:
def file_md5(path, chunk_size=1 << 20):
    digest = hashlib.md5()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


file_records = []
available = {}

for label, path in {**CHECKPOINTS, **EXTRA_CHECKPOINTS}.items():
    record = {"label": label, "path": str(path), "exists": path.exists()}
    if path.exists():
        available[label] = path
        record["md5"] = file_md5(path)
        record["size_mb"] = round(path.stat().st_size / 1e6, 1)
    file_records.append(record)

for label, path in {"repo manifest": get_path("manifest"), "Kaggle manifest": KAGGLE_MANIFEST}.items():
    record = {"label": label, "path": str(path), "exists": path.exists()}
    if path.exists():
        record["md5"] = file_md5(path)
        record["size_mb"] = round(path.stat().st_size / 1e6, 3)
    file_records.append(record)

files_table = pd.DataFrame(file_records).set_index("label")
print(files_table.to_string())

missing_reported = [label for label in CHECKPOINTS if label not in available]
assert not missing_reported, f"Reported checkpoints not found: {missing_reported}"

Expected: the two manifest copies have the same md5 (the repository manifest is the one used on Kaggle). The extra checkpoints may be missing if those Kaggle inputs are not attached.

In [ ]:
manifest = pd.read_csv(get_path("manifest"))
print("Repository manifest, rows per source / split / stage:")
print(manifest.groupby(["source", "split", "stage"]).size().to_string())

In [ ]:
SUMMARY_CONFIG_KEYS = [
    "training_stage",
    "learning_rate",
    "warmup_steps",
    "total_training_steps",
    "batch_size",
    "samples_per_track",
    "segment_frames",
    "max_sequence_length",
    "dropout",
    "random_seed",
    "codebook_loss_weights",
]

checkpoints = {}
summary_rows = []

for label, path in available.items():
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    checkpoints[label] = checkpoint

    config = checkpoint.get("config", {})
    history = checkpoint.get("history") or {}

    row = {
        "checkpoint": label,
        "top_level_keys": ", ".join(sorted(checkpoint.keys())),
        "epoch": checkpoint.get("epoch"),
        "global_step": checkpoint.get("global_step"),
        "batch_index": checkpoint.get("batch_index"),
        "best_val_loss": checkpoint.get("best_val_loss"),
        "history_epochs": len(history.get("epoch", [])),
        "parameters": sum(
            tensor.numel()
            for tensor in checkpoint["model_state_dict"].values()
        ),
    }

    for key in SUMMARY_CONFIG_KEYS:
        row[key] = config.get(key)

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index("checkpoint")
summary.to_csv(OUTPUT_DIR / "checkpoint_summary.csv")

with pd.option_context("display.max_colwidth", 80):
    print(summary.T.to_string())

In [ ]:
# Full configuration stored in each checkpoint.
for label, checkpoint in checkpoints.items():
    print(f"── {label} ──")
    print(json.dumps(checkpoint.get("config", {}), indent=2, default=str))

### Pairwise weight comparison

Maximum absolute difference between the parameters of every pair of checkpoints. A value of exactly 0 means the two files contain the same weights.

What to check:
- the three reported checkpoints must all differ from each other;
- `OST-init` should be identical to `Slakh-FT` (the OST fine-tune started from the evaluated Slakh-FT);
- if `OST-FT-v1` is identical to `Slakh-FT`, the first-submission generation script did not use OST fine-tuned weights.

In [ ]:
def max_abs_difference(state_a, state_b):
    if state_a.keys() != state_b.keys():
        return float("nan")
    return max(
        (state_a[key].float() - state_b[key].float()).abs().max().item()
        for key in state_a
    )


labels = list(checkpoints)
weight_difference = pd.DataFrame(index=labels, columns=labels, dtype=float)

for label_a in labels:
    for label_b in labels:
        weight_difference.loc[label_a, label_b] = max_abs_difference(
            checkpoints[label_a]["model_state_dict"],
            checkpoints[label_b]["model_state_dict"],
        )

weight_difference.to_csv(OUTPUT_DIR / "pairwise_max_abs_weight_difference.csv")
print(weight_difference.to_string(float_format=lambda value: f"{value:.3e}"))

identical_pairs = [
    (label_a, label_b)
    for index, label_a in enumerate(labels)
    for label_b in labels[index + 1:]
    if weight_difference.loc[label_a, label_b] == 0.0
]
print("\nIdentical weight pairs:", identical_pairs or "none")

reported = list(CHECKPOINTS)
duplicated_reported = [
    pair for pair in identical_pairs
    if pair[0] in reported and pair[1] in reported
]
assert not duplicated_reported, f"Reported checkpoints share weights: {duplicated_reported}"

### Training history stored in the checkpoints

Each `best` checkpoint stores the per-epoch history of its run, up to the epoch it was saved. Losses are weighted with the codebook weights of that run, so they are comparable within a run but not across runs.

Known limitation of the Baseline run: the resume logic reset `history` and `best_val_loss` at the start of every Kaggle session, so its stored history covers only the last session.

In [ ]:
HISTORY_COLUMNS = ["epoch", "global_step", "learning_rate", "train_loss", "val_loss"]
histories = {}

for label, checkpoint in checkpoints.items():
    history = checkpoint.get("history") or {}
    if not history.get("epoch"):
        print(f"{label}: no history stored in this checkpoint.")
        continue

    # Older checkpoints may miss some keys; lists are cut to a common length.
    present = [key for key in HISTORY_COLUMNS if history.get(key)]
    missing = [key for key in HISTORY_COLUMNS if key not in present]
    length = min(len(history[key]) for key in present)
    frame = pd.DataFrame({key: list(history[key])[:length] for key in present})
    if missing:
        print(f"{label}: history has no {missing}")

    if history.get("val_per_codebook_loss"):
        per_codebook = pd.DataFrame(
            list(history["val_per_codebook_loss"])[:length],
            columns=[f"val_Q{q}" for q in range(len(history["val_per_codebook_loss"][0]))],
        )
        frame = pd.concat([frame, per_codebook], axis=1)

    histories[label] = frame
    file_label = label.lower().replace("-", "_")
    frame.to_csv(OUTPUT_DIR / f"history_{file_label}.csv", index=False)

    print(f"── {label} ──")
    print(frame.to_string(index=False, float_format=lambda value: f"{value:.4g}"))

In [ ]:
if histories:
    figure, axes = plt.subplots(1, len(histories), figsize=(5 * len(histories), 4), squeeze=False)

    for axis, (label, frame) in zip(axes[0], histories.items()):
        x_values = frame["epoch"] if "epoch" in frame else frame.index
        for column, name in (("train_loss", "train"), ("val_loss", "validation")):
            if column in frame:
                axis.plot(x_values, frame[column], marker="o", label=name)
        axis.set_title(label)
        axis.set_xlabel("Epoch")
        axis.set_ylabel("Weighted cross-entropy (run-specific weights)")
        axis.legend()

    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "training_histories.png", dpi=150)
    plt.show()

### Step budget of each run

Needed for the data-versus-loss-weights ablation (E4): the new runs must use the same number of optimisation steps as the Slakh-FT run.

In [ ]:
budget_rows = []

for label, checkpoint in checkpoints.items():
    config = checkpoint.get("config", {})
    frame = histories.get(label)
    steps_per_epoch = None
    if frame is not None and len(frame) > 1 and "global_step" in frame:
        steps_per_epoch = sorted(set(np.diff(frame["global_step"]).tolist()))

    budget_rows.append({
        "checkpoint": label,
        "saved_epoch": checkpoint.get("epoch"),
        "global_step_at_save": checkpoint.get("global_step"),
        "planned_total_steps": config.get("total_training_steps"),
        "steps_per_epoch_observed": steps_per_epoch,
        "first_logged_epoch": None if frame is None or "epoch" not in frame else int(frame["epoch"].iloc[0]),
        "last_logged_epoch": None if frame is None or "epoch" not in frame else int(frame["epoch"].iloc[-1]),
    })

budget = pd.DataFrame(budget_rows).set_index("checkpoint")
budget.to_csv(OUTPUT_DIR / "step_budget.csv")
print(budget.to_string())

In [ ]:
provenance = {
    "files": file_records,
    "summary": json.loads(summary.to_json(orient="index")),
    "identical_weight_pairs": identical_pairs,
    "step_budget": json.loads(budget.to_json(orient="index")),
}

with open(OUTPUT_DIR / "provenance.json", "w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=2, default=str)

print("Saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)